
## Theory

Regression is a supervised machine learning technique used to predict continuous numerical values based on the relationship between dependent and independent variables.

**1. Simple Linear Regression:**
Simple Linear Regression predicts a dependent variable using a single independent variable. It represents the relationship between the variables using a straight-line equation:

y = b0 + b1x

where y is the predicted value, x is the independent variable, b0 is the intercept, and b1 is the regression coefficient (slope).

**2. Multiple Linear Regression:**
Multiple Linear Regression predicts a dependent variable using two or more independent variables. Its equation is:

y = b0 + b1x1 + b2x2 + ... + bnxn

The model calculates coefficients that describe the relationship between the input features and the target variable.

The performance of regression models can be evaluated using Mean Absolute Error (MAE), Mean Squared Error (MSE), and the coefficient of determination (R²).

In this practical, both algorithms are implemented from scratch using mathematical formulas to predict Data Warehouse marks from other subject marks.


In [1]:

import pandas as pd
import random
import math

# Load the processed dataset
df = pd.read_excel("EXPORTS/Mldata_Preprocessed.xlsx")

# Remove extra spaces from column names
df.columns = df.columns.str.strip()

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

df.head()


Dataset loaded successfully!
Dataset shape: (67, 9)

Column names:
['ROLL NO.', 'GENDER', 'SOFTWARE ENGINEER', 'TOC', 'COMPUTER NETWORK', 'MACHINE LEARNING', 'DATA WAREHOUSE', 'GRADE', 'RESULT']


,ROLL NO.,GENDER,SOFTWARE ENGINEER,TOC,COMPUTER NETWORK,MACHINE LEARNING,DATA WAREHOUSE,GRADE,RESULT
0,240104010001,M,0.500000,0.491525,0.482759,0.473684,0.454545,C,1
1,240104010002,F,0.500000,0.661017,0.741379,0.649123,0.545455,B,1
2,240104010003,M,0.633333,0.627119,0.620690,0.614035,0.600000,B,1
3,240104010004,M,0.983333,0.915254,0.948276,0.964912,0.890909,O,1
4,240104010005,M,0.500000,0.491525,0.482759,0.473684,0.454545,C,1


In [2]:

# Independent variables for simple linear regression
simple_feature = "SOFTWARE ENGINEER"

# Independent variables for multiple linear regression
multiple_features = [
    "SOFTWARE ENGINEER",
    "TOC",
    "COMPUTER NETWORK",
    "MACHINE LEARNING"
]

# Dependent variable
target = "DATA WAREHOUSE"

# Convert the selected columns to ordinary Python lists
X_simple = df[simple_feature].astype(float).tolist()

X_multiple = (
    df[multiple_features]
    .astype(float)
    .values
    .tolist()
)

y = df[target].astype(float).tolist()

print("Total records:", len(y))
print("Simple regression feature:", simple_feature)
print("Multiple regression features:", multiple_features)
print("Target variable:", target)


Total records: 67
Simple regression feature: SOFTWARE ENGINEER
Multiple regression features: ['SOFTWARE ENGINEER', 'TOC', 'COMPUTER NETWORK', 'MACHINE LEARNING']
Target variable: DATA WAREHOUSE


In [3]:

# Reproducible random shuffle
indices = list(range(len(y)))
random.Random(42).shuffle(indices)

split_index = int(0.80 * len(indices))

train_indices = indices[:split_index]
test_indices = indices[split_index:]

# Simple regression data
X_simple_train = [X_simple[i] for i in train_indices]
X_simple_test = [X_simple[i] for i in test_indices]

# Multiple regression data
X_multiple_train = [X_multiple[i] for i in train_indices]
X_multiple_test = [X_multiple[i] for i in test_indices]

# Target data
y_train = [y[i] for i in train_indices]
y_test = [y[i] for i in test_indices]

print("Training records:", len(train_indices))
print("Testing records:", len(test_indices))


Training records: 53
Testing records: 14


In [4]:

def fit_simple_linear_regression(X, Y):
    n = len(X)

    mean_x = sum(X) / n
    mean_y = sum(Y) / n

    numerator = sum(
        (X[i] - mean_x) * (Y[i] - mean_y)
        for i in range(n)
    )

    denominator = sum(
        (X[i] - mean_x) ** 2
        for i in range(n)
    )

    if denominator == 0:
        raise ValueError("Independent variable has no variation.")

    slope = numerator / denominator
    intercept = mean_y - slope * mean_x

    return intercept, slope


def predict_simple(intercept, slope, x):
    return intercept + slope * x


# Train the model
simple_intercept, simple_slope = fit_simple_linear_regression(
    X_simple_train, y_train
)

print("Simple Linear Regression Model")
print("Intercept:", simple_intercept)
print("Slope:", simple_slope)
print(
    f"Equation: y = {simple_intercept:.4f} "
    f"+ ({simple_slope:.4f} * x)"
)


Simple Linear Regression Model
Intercept: 0.1365685081825544
Slope: 0.7893059209847152
Equation: y = 0.1366 + (0.7893 * x)


In [5]:

def transpose(matrix):
    return [list(column) for column in zip(*matrix)]


def matrix_multiply(A, B):
    result = []

    for row in A:
        result_row = []

        for j in range(len(B[0])):
            value = sum(
                row[k] * B[k][j]
                for k in range(len(B))
            )
            result_row.append(value)

        result.append(result_row)

    return result


def solve_linear_system(A, b):
    # Create an augmented matrix
    n = len(A)
    augmented = [
        A[i][:] + [b[i]]
        for i in range(n)
    ]

    # Gaussian elimination with partial pivoting
    for col in range(n):
        pivot = max(
            range(col, n),
            key=lambda row: abs(augmented[row][col])
        )

        if abs(augmented[pivot][col]) < 1e-12:
            raise ValueError(
                "Normal equation is singular. "
                "Some features may be linearly dependent."
            )

        augmented[col], augmented[pivot] = (
            augmented[pivot], augmented[col]
        )

        pivot_value = augmented[col][col]

        for j in range(col, n + 1):
            augmented[col][j] /= pivot_value

        for row in range(n):
            if row != col:
                factor = augmented[row][col]

                for j in range(col, n + 1):
                    augmented[row][j] -= (
                        factor * augmented[col][j]
                    )

    return [augmented[i][n] for i in range(n)]


def fit_multiple_linear_regression(X, Y):
    # Add a column of ones for the intercept
    X_with_intercept = [
        [1.0] + row
        for row in X
    ]

    XT = transpose(X_with_intercept)

    # Calculate X-transpose-X
    XTX = matrix_multiply(XT, X_with_intercept)

    # Calculate X-transpose-y
    Y_column = [[value] for value in Y]
    XTY_matrix = matrix_multiply(XT, Y_column)
    XTY = [row[0] for row in XTY_matrix]

    # Solve (X-transpose-X) beta = X-transpose-y
    coefficients = solve_linear_system(XTX, XTY)

    return coefficients


def predict_multiple(coefficients, x):
    return coefficients[0] + sum(
        coefficients[i + 1] * x[i]
        for i in range(len(x))
    )


# Train the multiple regression model
multiple_coefficients = fit_multiple_linear_regression(
    X_multiple_train, y_train
)

print("Multiple Linear Regression Model")
print("Intercept:", multiple_coefficients[0])

for i, feature in enumerate(multiple_features):
    print(
        f"Coefficient for {feature}: "
        f"{multiple_coefficients[i + 1]:.6f}"
    )


Multiple Linear Regression Model
Intercept: 0.12641809888418462
Coefficient for SOFTWARE ENGINEER: 0.284840
Coefficient for TOC: 0.274233
Coefficient for COMPUTER NETWORK: 0.050199
Coefficient for MACHINE LEARNING: 0.193282


In [6]:

def calculate_metrics(actual, predicted):
    n = len(actual)

    mae = sum(
        abs(actual[i] - predicted[i])
        for i in range(n)
    ) / n

    mse = sum(
        (actual[i] - predicted[i]) ** 2
        for i in range(n)
    ) / n

    mean_actual = sum(actual) / n

    total_sum_squares = sum(
        (value - mean_actual) ** 2
        for value in actual
    )

    residual_sum_squares = sum(
        (actual[i] - predicted[i]) ** 2
        for i in range(n)
    )

    r2 = (
        1 - residual_sum_squares / total_sum_squares
        if total_sum_squares != 0
        else float("nan")
    )

    return mae, mse, r2


# Predictions for test records
simple_predictions = [
    predict_simple(simple_intercept, simple_slope, x)
    for x in X_simple_test
]

multiple_predictions = [
    predict_multiple(multiple_coefficients, x)
    for x in X_multiple_test
]

# Calculate metrics
simple_metrics = calculate_metrics(
    y_test, simple_predictions
)

multiple_metrics = calculate_metrics(
    y_test, multiple_predictions
)

results = pd.DataFrame({
    "Metric": ["MAE", "MSE", "R-squared"],
    "Simple Linear Regression": simple_metrics,
    "Multiple Linear Regression": multiple_metrics
})

print(results.to_string(index=False))


   Metric  Simple Linear Regression  Multiple Linear Regression
      MAE                  0.114247                    0.103247
      MSE                  0.020589                    0.022211
R-squared                  0.712722                    0.690089


In [8]:


# Original feature columns used during normalization
features = [
    "SOFTWARE ENGINEER",
    "TOC",
    "COMPUTER NETWORK",
    "MACHINE LEARNING",
    "DATA WAREHOUSE"
]

# Minimum and maximum values from the ORIGINAL dataset
# These must match the values used when fitting MinMaxScaler.
feature_min = {
    "SOFTWARE ENGINEER": 0,
    "TOC": 1,
    "COMPUTER NETWORK": 2,
    "MACHINE LEARNING": 3,
    "DATA WAREHOUSE": 5
}

feature_max = {
    "SOFTWARE ENGINEER": 60,
    "TOC": 60,
    "COMPUTER NETWORK": 60,
    "MACHINE LEARNING": 60,
    "DATA WAREHOUSE": 60
}

def minmax_normalize(subject, value):
    minimum = feature_min[subject]
    maximum = feature_max[subject]

    if maximum == minimum:
        return 0.0

    return (value - minimum) / (maximum - minimum)


print("Enter the student's original marks.\n")

raw_marks = {}

for subject in multiple_features:
    while True:
        try:
            marks = float(input(f"Enter {subject} marks: "))

            if marks < feature_min[subject] or marks > feature_max[subject]:
                print(
                    f"Enter marks between {feature_min[subject]} "
                    f"and {feature_max[subject]}."
                )
                continue

            raw_marks[subject] = marks
            break

        except ValueError:
            print("Please enter a valid numeric value.")

# Normalize inputs using the original dataset's min/max values
normalized_marks = {
    subject: minmax_normalize(subject, raw_marks[subject])
    for subject in multiple_features
}

print("\nNormalized Input Data:")
for subject, value in normalized_marks.items():
    print(f"{subject}: {value:.6f}")


Enter the student's original marks.


Normalized Input Data:
SOFTWARE ENGINEER: 0.766667
TOC: 0.898305
COMPUTER NETWORK: 0.327586
MACHINE LEARNING: 0.894737


In [9]:

# Input for simple linear regression
simple_input = normalized_marks["SOFTWARE ENGINEER"]

simple_prediction = predict_simple(
    simple_intercept,
    simple_slope,
    simple_input
)

# Input for multiple linear regression
multiple_input = [
    normalized_marks[feature]
    for feature in multiple_features
]

multiple_prediction = predict_multiple(
    multiple_coefficients,
    multiple_input
)

# Convert normalized predictions into approximate raw marks
simple_raw_prediction = simple_prediction * max_marks[target]
multiple_raw_prediction = multiple_prediction * max_marks[target]

print("\n--- Prediction Results ---")

print(
    f"Simple Linear Regression: "
    f"{simple_prediction:.4f} normalized "
    f"(approximately {simple_raw_prediction:.2f}/"
    f"{max_marks[target]} marks)"
)

print(
    f"Multiple Linear Regression: "
    f"{multiple_prediction:.4f} normalized "
    f"(approximately {multiple_raw_prediction:.2f}/"
    f"{max_marks[target]} marks)"
)



--- Prediction Results ---
Simple Linear Regression: 0.7417 normalized (approximately 40.79/55 marks)
Multiple Linear Regression: 0.7805 normalized (approximately 42.93/55 marks)
